# Deteccion de Rostros con OpenCV

Programa de vision artificial para detectar rostros en imagenes y video en tiempo real, compatible con OpenCV 4.x y 5.x.

## Paso 1: Instalacion y carga de librerias

In [ ]:
#1. Instalacion de OpenCV Library
#pip install opencv-python

#2. Importar librerias necesarias
import cv2
import os
import matplotlib.pyplot as plt

#3. Definir rutas de archivos
ruta_imagen = 'faces.jpg'
ruta_clasificador_xml = 'haarcascade_frontalface_default.xml'
ruta_modelo_yunet = 'face_detection_yunet.onnx'

#4. Cargar y verificar imagen
imagen_original = cv2.imread(ruta_imagen)
if imagen_original is None:
    raise FileNotFoundError(f'No se pudo cargar la imagen: {ruta_imagen}')

#5. Informacion de la imagen y versiones
print(f'Version OpenCV: {cv2.__version__}')
print(f'Dimensiones imagen: {imagen_original.shape}')

## Paso 2: Imagen en escala de grises

In [ ]:
#6. Convertir imagen a escala de grises
imagen_gris = cv2.cvtColor(imagen_original, cv2.COLOR_BGR2GRAY)

#7. Mostrar imagen en escala de grises
plt.imshow(imagen_gris, cmap='gray')
plt.axis('off')
plt.show()

## Paso 3: Cargar detector de rostros

In [ ]:
#8. Cargar detector de rostros (compatible OpenCV 4.x y 5.x)
detector_clasico_disponible = hasattr(cv2, 'CascadeClassifier')
clasificador_rostros = None
detector_yunet = None

if detector_clasico_disponible:
    clasificador_rostros = cv2.CascadeClassifier(ruta_clasificador_xml)
    if clasificador_rostros.empty():
        print('Advertencia: No se pudo cargar clasificador XML, se usara detector YuNet')
        clasificador_rostros = None
        detector_clasico_disponible = False

if not detector_clasico_disponible:
    if os.path.exists(ruta_modelo_yunet):
        altura, ancho = imagen_original.shape[:2]
        detector_yunet = cv2.FaceDetectorYN_create(
            ruta_modelo_yunet,
            '',
            (ancho, altura),
            0.6,
            0.3,
            5000
        )
        print('Detector YuNet (OpenCV 5.x) cargado correctamente')
    else:
        raise FileNotFoundError(
            'No hay detector disponible. Instale OpenCV<5 o descargue '
            'face_detection_yunet.onnx en la carpeta Visionartificial/'
        )

## Paso 4: Deteccion de rostros

In [ ]:
#9. Detectar rostros en la imagen
if clasificador_rostros is not None:
    rostros_detectados = clasificador_rostros.detectMultiScale(
        imagen_gris,
        scaleFactor=1.1,
        minNeighbors=8,
        minSize=(40, 40)
    )
    print(f'Clasificador clasico: {len(rostros_detectados)} rostro(s) detectado(s)')
else:
    _, resultados = detector_yunet.detect(imagen_original)
    rostros_detectados = []
    if resultados is not None:
        for det in resultados:
            x, y, w, h = int(det[0]), int(det[1]), int(det[2]), int(det[3])
            rostros_detectados.append([x, y, w, h])
    print(f'Detector YuNet: {len(rostros_detectados)} rostro(s) detectado(s)')

## Paso 5: Visualizar resultados

In [ ]:
#10. Dibujar rectangulos sobre los rostros detectados
for (x, y, w, h) in rostros_detectados:
    cv2.rectangle(imagen_original, (x, y), (x + w, y + h), (0, 0, 255), 2)

#11. Convertir BGR a RGB para Matplotlib
imagen_rgb = cv2.cvtColor(imagen_original, cv2.COLOR_BGR2RGB)

#12. Mostrar imagen final con rostros detectados
plt.figure(figsize=(20, 10))
plt.imshow(imagen_rgb)
plt.axis('off')
plt.show()

## Paso 6: Deteccion en tiempo real con camara

In [ ]:
#13. Acceder a camara y definir funcion de deteccion en video
captura_video = cv2.VideoCapture(0)

#14. Funcion para detectar rostros en flujo de video
def detectar_y_dibujar_rostros(cuadro_video):
    if clasificador_rostros is not None:
        imagen_gris_local = cv2.cvtColor(cuadro_video, cv2.COLOR_BGR2GRAY)
        rostros = clasificador_rostros.detectMultiScale(
            imagen_gris_local, 1.1, 8, minSize=(40, 40)
        )
    else:
        h_loc, w_loc = cuadro_video.shape[:2]
        detector_yunet.setInputSize((w_loc, h_loc))
        _, resultados = detector_yunet.detect(cuadro_video)
        rostros = []
        if resultados is not None:
            for det in resultados:
                x, y, w, h = int(det[0]), int(det[1]), int(det[2]), int(det[3])
                rostros.append([x, y, w, h])
    for (x, y, w, h) in rostros:
        cv2.rectangle(cuadro_video, (x, y), (x + w, y + h), (0, 0, 255), 2)
    return rostros

In [1]:
#15. Bucle principal de video en tiempo real
while True:
    lectura_exitosa, cuadro_actual = captura_video.read()
    if not lectura_exitosa:
        print('No se pudo leer el flujo de video')
        break
    detectar_y_dibujar_rostros(cuadro_actual)
    cv2.imshow('Rostros detectados en tiempo real', cuadro_actual)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
captura_video.release()
cv2.destroyAllWindows()

NameError: name 'captura_video' is not defined